In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, accuracy_score

# Load features
df = pd.read_csv('data/features.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

FEATURE_COLS = [
    'home_form_last5',
    'away_form_last5',
    'home_avg_scored',
    'home_avg_conceded',
    'away_avg_scored',
    'away_avg_conceded',
    'ref_matches',
    'ref_home_win_rate',
    'ref_yellows_avg',
    'ref_reds_avg',
    'ref_goals_avg',
    'elo_home',
    'elo_away',
    'elo_diff',
    'h2h_matches',
    'h2h_home_wins',
    'h2h_draws',
    'h2h_goals_avg',
]

X = df[FEATURE_COLS].values
y = df['target'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classes: {np.unique(y)}")

In [ ]:
# --- TRAIN WITH TIME SERIES SPLIT ---
tscv = TimeSeriesSplit(n_splits=5)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    probs = model.predict_proba(X_val)
    preds = model.predict(X_val)

    ll = log_loss(y_val, probs)
    acc = accuracy_score(y_val, preds)

    fold_results.append({'fold': fold + 1, 'log_loss': ll, 'accuracy': acc})
    print(f"Fold {fold + 1} → log_loss: {ll:.4f} | accuracy: {acc:.4f}")

results_df = pd.DataFrame(fold_results)
print(f"\nAverage log_loss: {results_df['log_loss'].mean():.4f}")
print(f"Average accuracy: {results_df['accuracy'].mean():.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Feature importance
importance = model.feature_importances_
feat_imp = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importance
}).sort_values('importance', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp['feature'], feat_imp['importance'])
ax.set_title('Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('data/feature_importance.png', dpi=150)
plt.show()

print(feat_imp.sort_values('importance', ascending=False).to_string())

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
import pickle

# Train final model on ALL data
final_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
)

# Calibrate probabilities
# FIX: Use TimeSeriesSplit instead of random KFold (cv=3).
# Random KFold leaks future data into the calibration step.
calibrated_model = CalibratedClassifierCV(
    final_model,
    method='isotonic',
    cv=TimeSeriesSplit(n_splits=3)
)

calibrated_model.fit(X, y)

# Save model
with open('data/model.pkl', 'wb') as f:
    pickle.dump(calibrated_model, f)

# Quick sanity check
sample = X[-5:]
probs = calibrated_model.predict_proba(sample)
sample_df = df.tail(5)[['Date', 'HomeTeam', 'AwayTeam']].copy()
sample_df['actual'] = y[-5:]  # 0=H, 1=D, 2=A
sample_df['prob_home'] = probs[:, 0].round(3)
sample_df['prob_draw'] = probs[:, 1].round(3)
sample_df['prob_away'] = probs[:, 2].round(3)

print(sample_df.to_string())
print("\n✅ Model saved to data/model.pkl")